In [2]:
import os
os.chdir("/storage/Subhadeep/HEIP/HEIP")

In [3]:
import torch
#import matplotlib.pyplot as plt
from pathlib import Path
from cellseg_models_pytorch.inference import SlidingWindowInferer
from cellseg_models_pytorch.utils import FileHandler
from src.unet import get_seg_model, convert_state_dict, MODEL_PARTS
from PIL import Image

/storage/Subhadeep/HEIP_Env/lib/python3.9/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: libtorch_cuda_cu.so: cannot open shared object file: No such file or directory
  warn(f"Failed to load image Python extension: {e}")
/storage/Subhadeep/HEIP_Env/lib/python3.9/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/storage/Subhadeep/HEIP_Env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
pip install cellseg-models-pytorch==0.1.16

INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 1.7 MB/s  0:00:01 eta 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 4.2 MB/s  0:00:00 eta 0:00:010m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 MB 3.3 MB/s  0:00:12m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 86.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 86.0 MB/s  0:00:00 eta 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 MB 88.3 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.0/21.0 MB 84.9 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 68.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.1/557.1 MB 80.2 MB/s  0:00:06m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 14.3 MB/s  0:00:00 e

In [6]:
import os
import torch

# Define paths
valid_img_root_dir = "/storage/Subhadeep/LC25000/LC25000/L"
save_root_dir = "/storage/Subhadeep/LC25000/LC25000_Segment_Full"
path_to_weights = "/storage/Subhadeep/HEIP/HEIP/last.ckpt"

# Initialize model
unet = get_seg_model()

# Initialize model
unet = get_seg_model()
ckpt_old = torch.load(path_to_weights, map_location=lambda storage, loc: storage, weights_only=False)
new_state_dict = convert_state_dict(MODEL_PARTS, unet.state_dict(), ckpt_old["state_dict"])
unet.load_state_dict(new_state_dict, strict=True)

# Determine device
#device = "cuda" if torch.cuda.is_available() else "cpu"

# Iterate over each subdirectory in valid_img_root_dir
for folder_name in os.listdir(valid_img_root_dir):
    valid_img_dir = os.path.join(valid_img_root_dir, folder_name)

    if os.path.isdir(valid_img_dir):  # Ensure it's a directory
        # Create corresponding save directory path
        folder_save_dir = os.path.join(save_root_dir, folder_name)

        # Skip if already processed
        if os.path.exists(folder_save_dir):
            print(f"Skipping {folder_name} (already processed).")
            continue

        # Create save directory
        os.makedirs(folder_save_dir, exist_ok=True)

        # Initialize and run sliding window inference
        inferer = SlidingWindowInferer(
            model=unet,
            input_folder=valid_img_dir,
            out_activations={"inst": "softmax", "type": "softmax", "omnipose": None},
            out_boundary_weights={"inst": False, "type": False, "omnipose": True},
            patch_size=(256, 256),
            stride=80,
            padding=120,
            instance_postproc="omnipose",
            batch_size=8,
            save_dir=folder_save_dir,
            #device=device
        )

        # Run inference
        inferer.infer()


Running inference: 100%|██████████| 625/625 [1:25:30<00:00,  8.21s/batch, Saving results to disk]


In [5]:
pip install pytorch-lightning==1.9.5

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.5/829.5 kB 63.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 92.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 70.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 888.0/888.0 MB 74.6 MB/s  0:00:09m0:00:0100:01
  Attempting uninstall: torch━━╺━━━━━━━━━━━━━━━━  7/12 [aiosignal]eballs]es]
    Found existing installation: torch 1.13.1m━━━━━━━━━━━━━━━━  7/12 [aiosignal]
    Uninstalling torch-1.13.1:━━━━╸━━━━━━━━━━━━━  8/12 [torch]
      Successfully uninstalled torch-1.13.1╸━━━━━━━━━━━━━  8/12 [torch]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [pytorch-lightning]pytorch-lightning]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cellseg-models-pytorch 0.1.16 requires torch<2.0.0,>=1.8.1, but you have torch 2.8.0 which is incompatible.
Note: you may n

In [5]:
pip install omegaconf==2.3.0

  Preparing metadata (setup.py) ... done
  DEPRECATION: Building 'antlr4-python3-runtime' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'antlr4-python3-runtime'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  Created wheel for antlr4-python3-runtime: filename=antlr4_python3_runtime-4.9.3-py3-none-any.whl size=144591 sha256=8ffcd58f4b9f7f39163bb2a9431309272eb89879387d4e936aa32b6b33ab9df4
  Stored in directory: /home/subhadeep/.cache/pip/wheels/23/cf/80/f3efa822e6ab23277902ee9165fe772eeb1dfb8014f359020a
Successfully built antlr4-python3-runtime
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [omegaconf]/2 [omegaconf]
Note: you may need to restart the kernel to use updated

In [6]:
import os
import torch

# Define paths
valid_img_root_dir = "/storage/Subhadeep/Rajiv_Skin_processing/Skin_Crop_Rajiv/CASE 32 SKIN"
save_root_dir = "/storage/Subhadeep/Rajiv_Skin_processing/Rajiv_Skin_Seg_HEIP"
path_to_weights = "/storage/Subhadeep/HEIP/HEIP/last.ckpt"

# Initialize model
unet = get_seg_model()

# Initialize model
unet = get_seg_model()
ckpt_old = torch.load(path_to_weights, map_location=lambda storage, loc: storage, weights_only=False)
new_state_dict = convert_state_dict(MODEL_PARTS, unet.state_dict(), ckpt_old["state_dict"])
unet.load_state_dict(new_state_dict, strict=True)

# Determine device
#device = "cuda" if torch.cuda.is_available() else "cpu"

# Iterate over each subdirectory in valid_img_root_dir
for folder_name in os.listdir(valid_img_root_dir):
    valid_img_dir = os.path.join(valid_img_root_dir, folder_name)

    if os.path.isdir(valid_img_dir):  # Ensure it's a directory
        # Create corresponding save directory path
        folder_save_dir = os.path.join(save_root_dir, folder_name)

        # Skip if already processed
        if os.path.exists(folder_save_dir):
            print(f"Skipping {folder_name} (already processed).")
            continue

        # Create save directory
        os.makedirs(folder_save_dir, exist_ok=True)

        # Initialize and run sliding window inference
        inferer = SlidingWindowInferer(
            model=unet,
            input_folder=valid_img_dir,
            out_activations={"inst": "softmax", "type": "softmax", "omnipose": None},
            out_boundary_weights={"inst": False, "type": False, "omnipose": True},
            patch_size=(256, 256),
            stride=80,
            padding=120,
            instance_postproc="omnipose",
            batch_size=8,
            save_dir=folder_save_dir,
            #device=device
        )

        # Run inference
        inferer.infer()


Running inference: 100%|██████████| 36/36 [10:27<00:00, 17.43s/batch, Saving results to disk]
